# Embedding Evaluation — Example Notebook

End-to-end walkthrough using `example_tight.json` (tight scenario) and `example_sparse.json` (sparse scenario).  
Run all cells top-to-bottom; all figures are interactive (Plotly).

**Tight scenario** — orthogonal class prototypes with tiny per-sample noise.  
Intra-class cosine ≈ 0.99, inter-class ≈ 0.00.  All retrieval KPIs are near-perfect.

**Sparse scenario** — nearby class prototypes (~60° apart) with large per-sample noise.  
Classes overlap heavily in embedding space — gap ≈ 0.07, purity@5 ≈ 0.56.

In [1]:
import json
import sys
from pathlib import Path

# Ensure the package is importable when running without `pip install -e .`
sys.path.insert(0, str(Path("..").resolve()))

from pai.ag_emb.services.evaluate import run_evaluation
from pai.ag_emb.services.reporting import (
    plot_cosine_similarity,
    plot_knn_confusion,
    plot_lle,
    plot_tsne,
    print_result,
)

/mnt/c/Users/MLomb/OneDrive/Desktop/ORS/pai-ag-emb/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load Data

In [2]:
# example_tight.json lives alongside this notebook in examples/
payload_path = Path("example_tight.json")
if not payload_path.exists():
    raise FileNotFoundError("example_tight.json not found — start Jupyter from the examples/ directory")

with open(payload_path) as f:
    payload = json.load(f)

embeddings: dict[str, list[float]] = payload["embeddings"]
dim = len(next(iter(embeddings.values())))
print(f"Loaded {len(embeddings)} embeddings  (dim={dim})")

Loaded 20 embeddings  (dim=16)


## Run Evaluation

In [3]:
result = run_evaluation(
    image_embeddings=embeddings,
    k_values=[5, 10],
    dataset_root=None,
    sample_pairs=None,
)

print_result(result)

Confusion matrix: 100%|██████████| 7/7 [00:00<00:00, 40.67step/s]     

n_items      : 20
embedding_dim: 16
classes      : ['corn', 'soybean']
k_values     : [5, 10]

── global_metrics ──────────────────────────────────────────────────────
  pairwise cosine    : mean=0.4731  std=0.4941  (p05=-0.0340  p50=0.0565  p95=0.9965)
  centroid cosine    : mean=0.7067  std=0.0154  norm=0.7067
  intra/inter gap    : 0.9885  (intra=0.9934  inter=0.0049)

  KNN purity@5      : mean=1.0000  std=0.0000
  KNN purity@10     : mean=0.9000  std=0.0000
  nDCG@5           : mean=1.0000  std=0.0000
  nDCG@10          : mean=1.0000  std=0.0000
  MAP@5            : mean=0.5556  std=0.0000
  MAP@10           : mean=1.0000  std=0.0000

  effective_rank     : 1.02  (ratio=0.0640,  dim=16)

── per_class ───────────────────────────────────────────────────────────

  [corn]  n=10
    pairwise cosine  : mean=0.9938  std=0.0022
    centroid cosine  : mean=0.9972  norm=0.9972
    effective_rank   : 5.30  (ratio=0.3311)
    KNN purity@5     : mean=1.0000  std=0.0000  (p05=1.0000  p95=1.000

## Visualizations

All plots are interactive — hover for details, click legend entries to toggle classes, and drag to rotate 3D views.

### KNN Confusion Matrix

Rows = true class, columns = neighbor class, values = fraction of k-NN neighbors belonging to each class.  
The diagonal equals mean KNN purity — higher is better.

In [4]:
plot_knn_confusion(result, output_path=None)

### Pairwise Cosine Similarity

Full N×N cosine similarity matrix sorted by class.  Within-class blocks sit on the diagonal — tighter, brighter blocks indicate a more discriminative embedding space.

In [5]:
plot_cosine_similarity(embeddings, result, output_path=None)

### t-SNE — 2D

t-SNE preserves local neighborhood structure.  Well-separated clusters indicate the model has learned class-discriminative features.

In [6]:
plot_tsne(embeddings, result, output_path=None, dimensions=2)

  t-SNE 2D — fitting 20 samples (perplexity=6, iter=1000)...


### t-SNE — 3D

3D variant — drag to rotate, scroll to zoom.

In [7]:
plot_tsne(embeddings, result, output_path=None, dimensions=3)

  t-SNE 3D — fitting 20 samples (perplexity=6, iter=2000)...


### LLE — 3D

Locally Linear Embedding preserves local geometry rather than global distances, complementing the t-SNE view.  Drag to rotate.

In [8]:
plot_lle(embeddings, result, output_path=None)

  LLE 3D — fitting 20 samples (n_neighbors=6)...


---

## Sparse Scenario

Same image paths and label structure, but class prototypes are close together (~60° apart) and per-sample noise is large.  
Compare these KPIs and plots directly against the tight scenario above to see how embedding quality degrades.

In [9]:
sparse_path = Path("example_sparse.json")
with open(sparse_path) as f:
    sparse_payload = json.load(f)

sparse_embeddings: dict[str, list[float]] = sparse_payload["embeddings"]
print(f"Loaded {len(sparse_embeddings)} sparse embeddings  (dim={len(next(iter(sparse_embeddings.values())))})")

Loaded 20 sparse embeddings  (dim=16)


In [10]:
sparse_result = run_evaluation(
    image_embeddings=sparse_embeddings,
    k_values=[5, 10],
    dataset_root=None,
    sample_pairs=None,
)

print_result(sparse_result)

Confusion matrix: 100%|██████████| 7/7 [00:00<00:00, 51.44step/s]     

n_items      : 20
embedding_dim: 16
classes      : ['corn', 'soybean']
k_values     : [5, 10]

── global_metrics ──────────────────────────────────────────────────────
  pairwise cosine    : mean=0.1602  std=0.2115  (p05=-0.1589  p50=0.1608  p95=0.5376)
  centroid cosine    : mean=0.4497  std=0.1425  norm=0.4497
  intra/inter gap    : 0.0712  (intra=0.1977  inter=0.1265)

  KNN purity@5      : mean=0.5600  std=0.1744
  KNN purity@10     : mean=0.5800  std=0.1470
  nDCG@5           : mean=0.5660  std=0.1966
  nDCG@10          : mean=0.6153  std=0.1606
  MAP@5            : mean=0.2282  std=0.1107
  MAP@10           : mean=0.4357  std=0.1777

  effective_rank     : 9.39  (ratio=0.5868,  dim=16)

── per_class ───────────────────────────────────────────────────────────

  [corn]  n=10
    pairwise cosine  : mean=0.2265  std=0.2039
    centroid cosine  : mean=0.5512  norm=0.5512
    effective_rank   : 6.05  (ratio=0.3784)
    KNN purity@5     : mean=0.6000  std=0.1549  (p05=0.3800  p95=0.800

### Sparse — KNN Confusion Matrix

In [11]:
plot_knn_confusion(sparse_result, output_path=None)

### Sparse — Pairwise Cosine Similarity

In [12]:
plot_cosine_similarity(sparse_embeddings, sparse_result, output_path=None)

### Sparse — t-SNE 2D

In [13]:
plot_tsne(sparse_embeddings, sparse_result, output_path=None, dimensions=2)

  t-SNE 2D — fitting 20 samples (perplexity=6, iter=1000)...


### Sparse — t-SNE 3D

In [14]:
plot_tsne(sparse_embeddings, sparse_result, output_path=None, dimensions=3)

  t-SNE 3D — fitting 20 samples (perplexity=6, iter=2000)...


### Sparse — LLE 3D

In [15]:
plot_lle(sparse_embeddings, sparse_result, output_path=None)

  LLE 3D — fitting 20 samples (n_neighbors=6)...
